# HTR pipeline for "Indiferente General 1544" (Tenerife Botanical Garden, late 18th – early 19th c.)

End-to-end [Kraken](https://github.com/mittagessen/kraken) workflow for a 248-folio archival dossier
(Archivo General de Indias): image preprocessing, line segmentation, line-by-line ground-truth
alignment, training from the CATMuS/McCATMuS base model (V3 and V4), evaluation on held-out folios,
application to the whole dossier, quality triage and text extraction.

| Section | Content |
|---|---|
| 0 | Configuration (all paths and parameters in one place) |
| 1 | Preprocessing: grayscale + mild CLAHE |
| 2 | Line segmentation (baseline segmenter, no OCR) |
| 3 | Ground-truth alignment: local tool to type the text of each detected line |
| 4–5 | V3: training from the base model and evaluation of the saved checkpoints |
| 6 | V4: second training round with a held-out test set and the honest evaluation |
| 7 | Applying V4 to the whole dossier (binarize, segment, recognise) |
| 8 | Quick quality sampling and automatic triage |
| 9 | Text extraction from the ALTO XML |

**Design decisions**

- **No rotation inside the script.** Folios are rotated and cropped by hand beforehand. The image treatment is
  grayscale + mild CLAHE (clip limit 1.5, 4x4 tiles); the first, more aggressive CLAHE amplified the ink
  show-through from the reverse side and made the segmenter over-split lines.
- **Each training round restarts from the base model** (`McCATMuS_nfd_nofix_V1`) rather than chaining
  V1 → V2 → V3, which carried inconsistencies from round to round.
- **The ALTO XML of the whole dossier is kept**, not only the plain text, so line coordinates remain available
  for citation and verification against the original.
- **One explicit manual step:** aligning the transcription with the detected lines (section 3). It is not
  automated, to avoid silent line/text misalignments.
- **A held-out test set** (5 folios never used in training) is set apart for V4 (section 6.2). `ketos train`
  validates on a split of the training pages, so it cannot reveal memorisation; `ketos test` on pages the model
  has never seen can.

**Results** (output of `ketos test` on the 5 held-out folios, section 6.5):

| Model | Character accuracy | Word accuracy |
|---|---|---|
| V3 | 50.99 % | 4.40 % |
| V4 | 85.23 % | 53.30 % |

Environment: Kraken 5.2.9, Python 3.10. Images and the intermediate image folders are not part of the repository.

## 0. Configuration

All paths and parameters in one place. Set `HTR_BASE_DIR` (or edit `BASE_DIR`) to the folder that holds your
images; every other folder is created below it.

In [ ]:
import glob
import os
import random
import re
import subprocess
from pathlib import Path

import cv2
from lxml import etree

BASE_DIR = Path(os.environ.get("HTR_BASE_DIR", "."))


def P(*parts):
    return str(BASE_DIR.joinpath(*parts))


# ---- INPUT IMAGES (not included in the repository) ----
IMAGES_FIRST_BATCH = P("transcribed_images")      # first hand-transcribed folios, already rotated and cropped
CORPUS_IMAGES = P("cropped_corpus")               # every folio of the dossier
IMAGES_V4_NEW = P("v4_new_pages")                 # additional training folios for V4
IMAGES_V4_TEST = P("v4_test_pages")               # held-out test folios (never used for training)

# ---- WORKING FOLDERS (created automatically) ----
CLAHE_FIRST_BATCH = P("v3_clahe_first_batch")
SEGMENTED_FIRST_BATCH = P("v3_segmented_for_alignment")
ALIGNED_XML = P("ground_truth_aligned")           # training ground truth (ALTO XML), written by the tool of section 3
CLAHE_CORPUS = P("clahe_corpus")
BINARIZED_CORPUS = P("binarized_corpus")
ALTO_FINAL_V4 = P("v4_alto_final")                # V4 transcription of the whole dossier (ALTO XML)
TXT_FINAL = P("dossier_v4.txt")

CLAHE_FIRST_BATCH_V4 = P("v4_clahe_first_batch")
BINARIZED_FIRST_BATCH_V4 = P("v4_binarized_first_batch")
CLAHE_V4_NEW = P("v4_clahe_new")
BINARIZED_V4_NEW = P("v4_binarized_new")
SEGMENTED_V4_NEW = P("v4_segmented_new")
CLAHE_V4_TEST = P("v4_clahe_test")
BINARIZED_V4_TEST = P("v4_binarized_test")
SEGMENTED_V4_TEST = P("v4_segmented_test")
ALIGNED_XML_TEST = P("ground_truth_test")         # held-out test ground truth: never read by the training cell

for folder in [CLAHE_FIRST_BATCH, SEGMENTED_FIRST_BATCH, ALIGNED_XML, CLAHE_CORPUS, BINARIZED_CORPUS,
               ALTO_FINAL_V4, CLAHE_FIRST_BATCH_V4, BINARIZED_FIRST_BATCH_V4, CLAHE_V4_NEW, BINARIZED_V4_NEW,
               SEGMENTED_V4_NEW, CLAHE_V4_TEST, BINARIZED_V4_TEST, SEGMENTED_V4_TEST, ALIGNED_XML_TEST]:
    os.makedirs(folder, exist_ok=True)

# ---- MODELS ----
BASE_MODEL = "McCATMuS_nfd_nofix_V1.mlmodel"      # public CATMuS-based model; not redistributed here
V3_NAME = "tenerife_V3"                           # ketos appends "_best.mlmodel" and checkpoint numbers
V4_NAME = "tenerife_V4"
MODEL_V3_FINAL = f"{V3_NAME}_best.mlmodel"
MODEL_V4_FINAL = f"{V4_NAME}_best.mlmodel"

# ---- PREPROCESSING PARAMETERS ----
# Milder than the original (3.0 / 8x8): that CLAHE amplified the ink show-through from the reverse
# side and the paper texture, which confused the segmenter.
# NOTE: if you change these values, run this cell FIRST and then re-run the cell that defines
# preprocess_image: Python fixes a function's default values when the function is defined.
CLAHE_CLIP_LIMIT = 1.5
CLAHE_TILE_GRID = (4, 4)

print("Configuration loaded.")

## 1. Preprocessing (CLAHE only, no rotation)

Grayscale + CLAHE, as a reusable function. Rotation and cropping are done by hand before this step.

In [ ]:
def preprocess_image(input_path, output_folder, clip_limit=CLAHE_CLIP_LIMIT, tile_grid=CLAHE_TILE_GRID):
    '''Grayscale + CLAHE. Does not rotate or crop (already done by hand).
    If the output image already exists (from an earlier session) it is not recomputed.'''
    name = os.path.basename(input_path)
    output_path = os.path.join(output_folder, f"clean_{name}")

    if os.path.exists(output_path):
        return output_path

    gray = cv2.imread(input_path, cv2.IMREAD_GRAYSCALE)
    if gray is None:
        raise ValueError(f"Could not read image: {input_path}")

    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    cv2.imwrite(output_path, clahe.apply(gray))
    return output_path


def preprocess_folder(input_folder, output_folder, pattern="*.jpg"):
    originals = sorted(glob.glob(os.path.join(input_folder, pattern)))
    print(f"Found {len(originals)} images in {input_folder}")

    cleaned, new = [], 0
    for path in originals:
        existed = os.path.exists(os.path.join(output_folder, f"clean_{os.path.basename(path)}"))
        try:
            cleaned.append(preprocess_image(path, output_folder))
            new += 0 if existed else 1
        except Exception as e:
            print(f"❌ Error with {path}: {e}")

    print(f"✔ {len(cleaned)}/{len(originals)} ready ({new} new, {len(cleaned) - new} already existed)")
    return cleaned


# Preprocess the first batch of hand-transcribed folios
first_batch_clean = preprocess_folder(IMAGES_FIRST_BATCH, CLAHE_FIRST_BATCH)

## 2. Segmentation (lines only, no OCR)

The XML with the line positions is produced with Kraken's generic baseline segmenter (`blla`), without text
yet. This XML is the input of the alignment tool of section 3, which shows each detected line and asks for its text.

In [ ]:
def segment_image(image_path, output_xml_folder):
    '''Runs baseline segmentation without OCR and without binarizing, so that the XML always
    references the real (preprocessed) image and never a temporary file that kraken creates and
    deletes when it binarizes.'''
    base = Path(image_path).stem
    xml_path = os.path.join(output_xml_folder, f"{base}.xml")

    if os.path.exists(xml_path):
        print(f"↷ {base}.xml already existed, skipped")
        return xml_path

    result = subprocess.run(
        ["kraken", "-a", "-i", image_path, xml_path, "segment", "-bl"],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"❌ Error segmenting {base}:\n{result.stderr}")
        return None

    print(f"✔ {base}.xml")
    return xml_path


first_batch_xml = [segment_image(p, SEGMENTED_FIRST_BATCH) for p in first_batch_clean]
first_batch_xml = [p for p in first_batch_xml if p]   # drop failures
print(f"\n{len(first_batch_xml)} folios segmented, ready to align.")

## 3. Ground-truth alignment: local tool

No account or server needed. The cell shows the crop of every line detected by the segmenter and lets you type
the corresponding text while looking at the crop. Each line is written straight into the XML.

- Transcribing a whole folio into a separate `.txt` first and pasting it here is transcribing the same text
  twice; read each crop and type directly.
- If a crop is ambiguous (a letter cut at the edge, an abbreviation that needs the whole sentence), keeping the
  full folio image open next to it helps, as an occasional reference rather than a source to copy from.
- You can stop at any time (Kernel → Interrupt) and resume later: folios already saved in `ALIGNED_XML` are
  skipped the next time.
- A blank answer (Enter, or a single space) **removes** that line from the output XML instead of saving it empty:
  a line without text is useless as training data and makes `ketos` warn for each one.
- Try it first on **one folio** and check that the crops are well framed. If a crop is cut or includes the
  neighbouring line, increase `margin` in `get_bbox_from_polygon`.
- The result is interchangeable with what eScriptorium produces (same kind of aligned ALTO XML).

In [ ]:
from PIL import Image
from IPython.display import display


def get_bbox_from_polygon(points_str, margin=15):
    '''Turns the POINTS attribute of a <Polygon> into a box (left, top, right, bottom). It extracts every
    number in the string and pairs them up (x, y), instead of assuming a separator between coordinates.'''
    numbers = list(map(float, re.findall(r'-?\d+\.?\d*', points_str)))
    xs, ys = numbers[0::2], numbers[1::2]
    return (max(0, min(xs) - margin), max(0, min(ys) - margin), max(xs) + margin, max(ys) + margin)


def align_page_manually(image_path, xml_in_path, xml_out_path):
    '''Shows every detected line as a crop and asks for its text. Writes a new XML with the text
    inserted; the input XML is not modified. A blank answer deletes the line.'''
    tree = etree.parse(xml_in_path)
    root = tree.getroot()
    ns_uri = etree.QName(root).namespace   # None if the XML has no namespace

    lines = tree.xpath('//*[local-name()="TextLine"]')
    image = Image.open(image_path)

    print(f"\n=== {Path(image_path).name} — {len(lines)} lines detected ===")
    discarded = 0
    for i, line in enumerate(lines, 1):
        polygon = line.xpath('.//*[local-name()="Polygon"]/@POINTS')
        if not polygon:
            print(f"[line {i}] no polygon, skipped.")
            continue

        display(image.crop(get_bbox_from_polygon(polygon[0])))
        text = input(f"Line {i}/{len(lines)} — text (Enter to discard the line): ")

        if text.strip() == "":
            line.getparent().remove(line)
            discarded += 1
            continue

        tag_string = f"{{{ns_uri}}}String" if ns_uri else "String"
        string_el = line.find(f".//{tag_string}")
        if string_el is None:
            string_el = etree.SubElement(line, tag_string)
        string_el.set("CONTENT", text)

    tree.write(xml_out_path, xml_declaration=True, encoding="UTF-8")
    print(f"✔ Saved: {xml_out_path} ({discarded} blank lines discarded)")

In [ ]:
# --- Try ONE folio first ---
test_output = os.path.join(ALIGNED_XML, "TRIAL_" + Path(first_batch_clean[0]).stem + ".xml")
align_page_manually(first_batch_clean[0], first_batch_xml[0], test_output)

In [ ]:
# --- When the trial looks right, run this for the whole first batch ---
# Folios that already have an aligned XML in ALIGNED_XML are skipped automatically.
for img_path, xml_path in zip(first_batch_clean, first_batch_xml):
    base = Path(img_path).stem
    out_path = os.path.join(ALIGNED_XML, f"{base}.xml")
    if os.path.exists(out_path):
        print(f"Already aligned: {base} (delete it to redo)")
        continue
    align_page_manually(img_path, xml_path, out_path)

print("\nDone. Continue with section 4 (training).")

### 3b. Remove blank lines from already aligned files

Blank lines left in an earlier version of the tool were saved as empty text instead of being removed. This cell
removes them from the files in `ALIGNED_XML`. Run it once (again only if you used the older version of the tool).

In [ ]:
def remove_empty_lines(xml_path):
    '''Removes the lines whose text is empty or only whitespace. Overwrites the file.
    Returns how many lines were removed.'''
    tree = etree.parse(xml_path)
    removed = 0
    for line in tree.xpath('//*[local-name()="TextLine"]'):
        content = line.xpath('.//*[local-name()="String"]/@CONTENT')
        text = content[0] if content else ""
        if text.strip() == "":
            line.getparent().remove(line)
            removed += 1
    if removed:
        tree.write(xml_path, xml_declaration=True, encoding="UTF-8")
    return removed


aligned_files = [a for a in glob.glob(os.path.join(ALIGNED_XML, "*.xml"))
                 if not os.path.basename(a).startswith("TRIAL_")]

total_removed = 0
for path in aligned_files:
    n = remove_empty_lines(path)
    total_removed += n
    if n:
        print(f"{os.path.basename(path)}: {n} blank lines removed")

print(f"\nTotal: {total_removed} blank lines removed from {len(aligned_files)} folios.")

### 3c. Fix the image reference in the aligned XML

XML files aligned with an earlier version of section 2 (which included `binarize`) pointed to a temporary file
instead of the real image, so `ketos train` could not find it. This cell rewrites that reference.

In [ ]:
def fix_image_reference(xml_path, real_image_path):
    '''Rewrites the <fileName> of the XML so that it points to the real image, not to the temporary
    file that kraken generated when it binarized.'''
    tree = etree.parse(xml_path)
    elements = tree.xpath('//*[local-name()="fileName"]')
    if not elements:
        return False
    for el in elements:
        el.text = real_image_path
    tree.write(xml_path, xml_declaration=True, encoding="UTF-8")
    return True


aligned_files = [a for a in glob.glob(os.path.join(ALIGNED_XML, "*.xml"))
                 if not os.path.basename(a).startswith("TRIAL_")]

fixed, missing = 0, []
for xml_path in aligned_files:
    base = Path(xml_path).stem                 # e.g. clean_IMG_4955
    candidates = glob.glob(os.path.join(CLAHE_FIRST_BATCH, f"{base}.*"))
    if not candidates:
        missing.append(base)
        continue
    if fix_image_reference(xml_path, candidates[0]):
        fixed += 1

print(f"{fixed}/{len(aligned_files)} XML files fixed with the real image path.")
if missing:
    print("⚠️ No real image found for:", missing)

## 4. V3 training: one clean run from the base model

Instead of the earlier chained rounds (V1 → V2 → V3), training restarts from the base model with the **whole**
first batch at once, so that `ketos` splits training and validation over the complete set.

In [ ]:
training_files = sorted(glob.glob(os.path.join(ALIGNED_XML, "*.xml")))
print(f"{len(training_files)} aligned folios found for training.")

# ketos shows a rich progress bar (Rich); run the command with the shell magic (!) rather than
# subprocess so that the interactive output is not lost.
#
# --workers 0 forces data loading in a single process. With several worker processes kraken can hit a
# race condition on temporary files (the typical "FileNotFoundError ... DataLoader worker"). With so few
# folios there is no speed penalty.
command = "ketos train -i {} -f alto --resize add --workers 0 -o {} {}".format(
    BASE_MODEL, V3_NAME, " ".join(f'"{a}"' for a in training_files)
)
print(command)

In [ ]:
# Uncomment and run once you have reviewed the command above:
# !{command}

## 5. Evaluating the saved checkpoints with `ketos test`

`ketos test` re-evaluates any trained model against aligned folios; it is pure inference, much faster than
retraining, and useful to recover the metrics if a training run was cut short.

> **Caution:** here the checkpoints are evaluated on the *same* folios they were trained on, so these numbers
> are optimistic (in-sample). The honest, held-out evaluation is in section 6.5.

In [ ]:
checkpoint_paths = sorted(glob.glob(f"{V3_NAME}_*.mlmodel"))
print(f"{len(checkpoint_paths)} models found:")
for r in checkpoint_paths:
    print(" -", Path(r).name)

eval_xml_files = sorted(
    a for a in glob.glob(os.path.join(ALIGNED_XML, "*.xml"))
    if not os.path.basename(a).startswith("TRIAL_")
)


def evaluate_model(model_path, xml_files):
    '''Runs ketos test on a model and returns its full report (text).'''
    result = subprocess.run(
        ["ketos", "test", "-f", "alto", "-m", model_path, *xml_files],
        capture_output=True, text=True
    )
    return result.stdout + result.stderr

In [ ]:
# ketos test on every checkpoint. With ~17 checkpoints it can take several minutes.
summary = []
for model_path in checkpoint_paths:
    print(f"Evaluating {Path(model_path).name}...")
    output = evaluate_model(model_path, eval_xml_files)
    accuracy_line = next((l for l in output.splitlines() if "Accuracy" in l), None)
    summary.append((Path(model_path).name, accuracy_line or "(could not read)"))

print("\n=== Summary ===")
for name, line in summary:
    print(f"{name}: {line}")

# V4: second training round

V4 combines the V3 folios (re-processed to match the current pipeline: mild CLAHE + binarization) with a new
batch of folios chosen for **variety** (botanical-inventory register, strong show-through, other hands), not
for order. It trains from the base model again, not from V3, to obtain a clean and consistent run.

## 6.1 Re-process the V3 folios with the current recipe

The manual alignment does not need to be redone: the text and the line positions do not change. Only the image is
regenerated (mild CLAHE + binarization) and each aligned XML is updated to point to it.

In [ ]:
def binarize_image(input_path, output_folder):
    '''Runs `kraken binarize` into a real, persistent file (kraken forces PNG anyway, because JPEG
    cannot store 1-bit-per-pixel images).'''
    base = Path(input_path).stem
    output_path = os.path.join(output_folder, f"{base}.png")
    if os.path.exists(output_path):
        return output_path
    result = subprocess.run(
        ["kraken", "-i", input_path, output_path, "binarize"],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"❌ Error binarizing {base}:\n{result.stderr}")
        return None
    return output_path


def binarize_folder(input_folder, output_folder):
    inputs = sorted(glob.glob(os.path.join(input_folder, "*.jpg")))
    binarized, new = [], 0
    for path in inputs:
        existed = os.path.exists(os.path.join(output_folder, f"{Path(path).stem}.png"))
        r = binarize_image(path, output_folder)
        if r:
            binarized.append(r)
            new += 0 if existed else 1
    print(f"✔ {len(binarized)}/{len(inputs)} binarized ({new} new, {len(binarized) - new} already existed)")
    return binarized


first_batch_clahe_v4 = preprocess_folder(IMAGES_FIRST_BATCH, CLAHE_FIRST_BATCH_V4)
first_batch_binarized_v4 = binarize_folder(CLAHE_FIRST_BATCH_V4, BINARIZED_FIRST_BATCH_V4)

# Point every aligned XML of the first batch to the new binarized image
fixed, missing = 0, []
for xml_path in glob.glob(os.path.join(ALIGNED_XML, "*.xml")):
    if os.path.basename(xml_path).startswith("TRIAL_"):
        continue
    base = Path(xml_path).stem
    candidates = glob.glob(os.path.join(BINARIZED_FIRST_BATCH_V4, f"{base}.*"))
    if not candidates:
        missing.append(base)
        continue
    if fix_image_reference(xml_path, candidates[0]):
        fixed += 1

print(f"{fixed} XML files of the first batch updated to the binarized image.")
if missing:
    print("⚠️ No binarized image found for:", missing)

## 6.2 Two new batches: training and an honest test set

With only the V3 folios, `ketos train` validates against a slice of those same folios, so it would never warn us
that the model memorises instead of generalising. That is exactly what happened: 23 of the 26 folios that the
first triage flagged as "legible" in the whole dossier were the 23 training folios. For V4 a small group is
kept that **never** enters `ketos train`, so that generalisation can be measured with `ketos test`.

1. Choose **20–25 new folios to TRAIN on** (with variety) and **3–5 folios to TEST on** (varied too, not the
   easiest ones).
2. Rotate and crop both batches by hand, as for the originals.
3. Type the text directly in the alignment tool (sections 6.3 and 6.4).
4. Keep each batch in its own folder: the test folios go through a separate alignment flow and output folder, so
   they cannot slip into training by accident.

In [ ]:
# Training batch: same pipeline as the first batch, now including binarization
new_clahe = preprocess_folder(IMAGES_V4_NEW, CLAHE_V4_NEW)
new_binarized = binarize_folder(CLAHE_V4_NEW, BINARIZED_V4_NEW)
new_xml = [segment_image(r, SEGMENTED_V4_NEW) for r in new_binarized]
new_xml = [r for r in new_xml if r]
print(f"\n{len(new_xml)} new folios segmented, ready to align.")

# Test batch: same pipeline, completely separate folders
test_clahe = preprocess_folder(IMAGES_V4_TEST, CLAHE_V4_TEST)
test_binarized = binarize_folder(CLAHE_V4_TEST, BINARIZED_V4_TEST)
test_xml = [segment_image(r, SEGMENTED_V4_TEST) for r in test_binarized]
test_xml = [r for r in test_xml if r]
print(f"\n{len(test_xml)} TEST folios segmented, ready to align.")

## 6.3 Align the new training folios (same tool as section 3)

The resulting XML files are saved straight into `ALIGNED_XML`, together with the V3 folios updated in 6.1.

In [ ]:
for img_path, xml_path in zip(new_binarized, new_xml):
    base = Path(img_path).stem
    out_path = os.path.join(ALIGNED_XML, f"{base}.xml")
    if os.path.exists(out_path):
        print(f"Already aligned: {base} (delete it to redo)")
        continue
    align_page_manually(img_path, xml_path, out_path)

print("\nDone. Continue with training (6.5).")

## 6.4 Align the TEST folios

Same as 6.3, but the result goes to `ALIGNED_XML_TEST`, a folder that the training cell never reads.

In [ ]:
for img_path, xml_path in zip(test_binarized, test_xml):
    base = Path(img_path).stem
    out_path = os.path.join(ALIGNED_XML_TEST, f"{base}.xml")
    if os.path.exists(out_path):
        print(f"Already aligned: {base} (delete it to redo)")
        continue
    align_page_manually(img_path, xml_path, out_path)

print("\nDone. The test batch stays apart from training.")

## 6.5 Train V4 and evaluate it honestly

All the training ground truth (the V3 folios re-processed plus the new ones: **58 folios**) is now together in
`ALIGNED_XML`. One clean run from the base model, as for V3. The test folios in `ALIGNED_XML_TEST` are never
read by the training cell.

In [ ]:
training_files_v4 = sorted(
    a for a in glob.glob(os.path.join(ALIGNED_XML, "*.xml"))
    if not os.path.basename(a).startswith("TRIAL_")
)
print(f"{len(training_files_v4)} folios in total to train V4.")

command_v4 = "ketos train -i {} -f alto --resize add --workers 0 -o {} {}".format(
    BASE_MODEL, V4_NAME, " ".join(f'"{a}"' for a in training_files_v4)
)
print(command_v4)

In [ ]:
# Review the command above and uncomment when ready:
# !{command_v4}

In [ ]:
# Honest evaluation: ketos test on the held-out folios, which no model has seen.
# V3 is evaluated on the same set, so the comparison is fair in both directions.
test_files = sorted(glob.glob(os.path.join(ALIGNED_XML_TEST, "*.xml")))
print(f"{len(test_files)} test folios, never seen by any model.\n")

for model_name, model_path in [("V3", MODEL_V3_FINAL), ("V4", MODEL_V4_FINAL)]:
    print(f"=== {model_name} ({model_path}) ===")
    output = evaluate_model(model_path, test_files)
    for line in output.splitlines():
        if "Accuracy" in line:
            print(line)
    print()

# Result obtained on the 5 held-out folios:
#   V3: 50.99 % Character Accuracy, 4.40 % Word Accuracy
#   V4: 85.23 % Character Accuracy, 53.30 % Word Accuracy

## 7. Apply V4 to the whole dossier

Mild CLAHE to even out shadows, then binarization as its own persistent file (not chained inside the same
kraken command, which avoids the temporary-file bug that broke training earlier). Warnings such as
`jpeg does not support 1bpp images... Forcing to png` are expected, because the binarized images are saved as PNG.

With ~250 folios on CPU this can take hours. It can be interrupted at any time: files that already exist are
skipped when it is resumed.

In [ ]:
corpus_clean = preprocess_folder(CORPUS_IMAGES, CLAHE_CORPUS)
corpus_binarized = binarize_folder(CLAHE_CORPUS, BINARIZED_CORPUS)


def transcribe_image(image_path, output_xml_folder, model):
    '''Segmentation + OCR with the trained model on an already binarized image.
    If the output XML exists it is reused.'''
    base = Path(image_path).stem
    xml_path = os.path.join(output_xml_folder, f"{base}.xml")
    if os.path.exists(xml_path):
        print(f"↷ {base}.xml already existed, skipped")
        return xml_path
    result = subprocess.run(
        ["kraken", "-a", "-i", image_path, xml_path, "segment", "-bl", "ocr", "-m", model],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"❌ Error transcribing {base}:\n{result.stderr}")
        return None
    print(f"✔ {base}.xml")
    return xml_path


alto_v4_paths = [transcribe_image(r, ALTO_FINAL_V4, MODEL_V4_FINAL) for r in corpus_binarized]
alto_v4_paths = [r for r in alto_v4_paths if r]
print(f"\n{len(alto_v4_paths)} folios transcribed with V4.")

## 8. Quality sampling and automatic triage

### 8.1 Quick sample

Prints the text of 5 random folios. Compare them against the original images to judge whether the recipe really
improved things, not just the line count.

In [ ]:
available_xml = glob.glob(os.path.join(ALTO_FINAL_V4, "*.xml"))
print(f"{len(available_xml)} folios transcribed so far.")

for xml_path in random.sample(available_xml, min(5, len(available_xml))):
    tree = etree.parse(xml_path)
    contents = tree.xpath('//*[local-name()="String"]/@CONTENT')
    print(f"\n=== {Path(xml_path).stem} ({len(contents)} words) ===")
    print(" ".join(contents)[:500])

### 8.2 Triage: which folios to review and which to transcribe by hand

A first version based on ~90 function words was too lax: clearly bad folios scored almost as high as truly
good ones. This version uses a real Spanish dictionary (`pyspellchecker`, `pip install pyspellchecker`) to
measure what share of the recognised words are valid words.

It remains an approximation: 18th-century Spanish spells differently from modern Spanish (e.g. "qual" for
"cual"), so some genuine words will count as unrecognised. The cell prints the ratio of reference folios that
were checked by eye (good and bad ones) so that the threshold `THRESHOLD` can be calibrated on evidence.

In [ ]:
from spellchecker import SpellChecker

spell = SpellChecker(language="es")

# Folios checked by eye, to calibrate the threshold on evidence
REFERENCE_QUALITY = {
    "clean_IMG_4975": "good (genuinely legible)",
    "clean_IMG_5021": "bad (sentence structure but twisted words)",
    "clean_IMG_4998": "bad (largely illegible)",
    "clean_IMG_5093": "medium-bad",
    "clean_IMG_4983": "mixed (real words mixed with noise)",
}


def legibility_ratio(xml_path):
    '''Share of words (length >= 3) that are valid Spanish words according to the dictionary.
    Approximate: see the notes above.'''
    tree = etree.parse(xml_path)
    contents = tree.xpath('//*[local-name()="String"]/@CONTENT')
    tokens = []
    for c in contents:
        tokens.extend(re.findall(r"[a-záéíóúñü]+", c.lower()))

    relevant = [t for t in tokens if len(t) >= 3]     # 1-2 character fragments are almost always noise
    if not relevant:
        return 0.0, len(tokens)

    known = spell.known(relevant)
    return sum(1 for t in relevant if t in known) / len(relevant), len(relevant)


results = []
for xml_path in sorted(glob.glob(os.path.join(ALTO_FINAL_V4, "*.xml"))):
    ratio, n_words = legibility_ratio(xml_path)
    results.append((Path(xml_path).stem, ratio, n_words))

print("=== Reference folios (to calibrate THRESHOLD) ===")
for name, ratio, n in results:
    if name in REFERENCE_QUALITY:
        print(f"{name}: {ratio:.2f}  -- {REFERENCE_QUALITY[name]}")

# Adjust using the table above: it should sit between the highest ratio of the "bad" folios
# and the lowest of the "good" ones.
THRESHOLD = 0.35

to_review = [r for r in results if r[1] >= THRESHOLD]
to_transcribe_by_hand = [r for r in results if r[1] < THRESHOLD]

print(f"\n{len(to_review)} folios to REVIEW AND CORRECT (probably legible)")
print(f"{len(to_transcribe_by_hand)} folios to TRANSCRIBE BY HAND (probably badly damaged)")

out_folder = os.path.dirname(TXT_FINAL) or "."
with open(os.path.join(out_folder, "triage_review_v4.txt"), "w", encoding="utf-8") as f:
    for name, ratio, n in sorted(to_review, key=lambda x: -x[1]):
        f.write(f"{name}\t{ratio:.2f}\t{n} words\n")

with open(os.path.join(out_folder, "triage_manual_v4.txt"), "w", encoding="utf-8") as f:
    for name, ratio, n in sorted(to_transcribe_by_hand, key=lambda x: x[1]):
        f.write(f"{name}\t{ratio:.2f}\t{n} words\n")

print(f"\nLists saved in {out_folder}: triage_review_v4.txt and triage_manual_v4.txt")

## 9. Extract the consolidated text from the ALTO XML

The extraction reads the `CONTENT` attribute of the ALTO `String` elements (one per word) of every `TextLine`,
using `local-name()` so that it does not depend on the exact namespace version of the Kraken in use. Test it on one file first: if the list comes
back empty, open one of the XML files and check the name of the tag that holds the recognised text.

In [ ]:
def extract_text_from_alto(xml_path):
    '''Returns the list of line texts (in order) of a Kraken ALTO XML file. In ALTO every
    TextLine holds one <String> per recognised word, so the words of each line are joined.'''
    tree = etree.parse(xml_path)
    lines = []
    for text_line in tree.xpath('//*[local-name()="TextLine"]'):
        words = text_line.xpath('./*[local-name()="String"]/@CONTENT')
        text = " ".join(words).strip()
        if text:
            lines.append(text)
    return lines


# --- try a single file first ---
if alto_v4_paths:
    sample = extract_text_from_alto(alto_v4_paths[0])
    print(f"Test on {alto_v4_paths[0]}: {len(sample)} lines found.")
    print(sample[:5])

In [ ]:
# When the cell above returns meaningful lines, write the whole dossier:
with open(TXT_FINAL, "w", encoding="utf-8") as f_out:
    for xml_path in sorted(alto_v4_paths):
        f_out.write("=" * 43 + "\n")
        f_out.write(f"FILE: {Path(xml_path).stem}\n")
        f_out.write("=" * 43 + "\n\n")
        for line in extract_text_from_alto(xml_path):
            f_out.write(line + "\n")
        f_out.write("\n\n")

print(f"Consolidated dossier written to: {TXT_FINAL}")
print("The ALTO XML is kept separately, with line coordinates for every folio.")

---
### Reproducibility

To record the library versions of the environment, inside your `kraken` environment:

```bash
conda env export --from-history > environment.yml
```